In [ ]:
import parselmouth
from parselmouth.praat import call
import librosa
import numpy as np
import os
from datetime import datetime
import csv
import re
import glob

def analyze_voice_biomarkers(audio_path):
    """
    Analyzes voice biomarkers from an audio file to assess stress and fatigue indicators.
    Returns a dictionary of voice biomarkers including Jitter, Shimmer, HNR, Formants, and MFCCs.
    """
    print(f"=== Analyzing Voice Biomarkers for {audio_path} ===")
    try:
        # --- Part 1: Analysis with Librosa ---
        y, sr = librosa.load(audio_path)
        
        # Extract MFCCs
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        mfccs_mean = np.mean(mfccs.T, axis=0)
        
        # --- Part 2: Clinical Analysis with Parselmouth (Praat) ---
        sound = parselmouth.Sound(audio_path)
        
        pointProcess = call(sound, "To PointProcess (periodic, cc)", 75, 600)
        
        # Calculate Jitter (frequency variation)
        # Mathematical formula in the background: $Jitter = \frac{\frac{1}{N-1} \sum_{i=1}^{N-1} |T_i - T_{i+1}|}{\frac{1}{N} \sum_{i=1}^{N} T_i}$
        localJitter = call(pointProcess, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3)
        
        # Calculate Shimmer (amplitude variation)
        localShimmer = call([sound, pointProcess], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6)
        
        # Calculate HNR (Harmonics-to-Noise Ratio)
        harmonicity = call(sound, "To Harmonicity (cc)", 0.01, 75, 0.1, 1.0)
        hnr = call(harmonicity, "Get mean", 0, 0)

        # --- Part 3: Formant Analysis ---
        formant = call(sound, "To Formant (burg)", 0.0, 5, 5500, 0.025, 50)
        f1_mean = call(formant, "Get mean", 1, 0, 0, "hertz")
        f2_mean = call(formant, "Get mean", 2, 0, 0, "hertz")
        
        print(f"Jitter Index: {localJitter * 100:.3f}% (Normal is typically < 1.04%)")
        print(f"Shimmer Index: {localShimmer * 100:.3f}% (Normal is typically < 3.8%)")
        print(f"HNR: {hnr:.2f} dB (Normal is typically > 20 dB)")
        print(f"Mean Formant 1 (F1): {f1_mean:.2f} Hz")
        print(f"Mean Formant 2 (F2): {f2_mean:.2f} Hz")
        
        return {
            "Jitter": localJitter,
            "Shimmer": localShimmer,
            "HNR": hnr,
            "F1": f1_mean,
            "F2": f2_mean,
            "MFCCs": mfccs_mean.tolist() # Convert numpy array to list for JSON serialization
        }
    except Exception as e:
        print(f"An error occurred during voice analysis for {audio_path}: {e}")
        # This can happen if the audio file is not found or is corrupted.
        return None

def save_voice_to_csv(person_id, analysis_results, file_name='../../data/processed/voice_analysis.csv'):
    """
    Saves the voice analysis results to a CSV file.
    """
    file_exists = os.path.isfile(file_name)
    with open(file_name, 'a', newline='') as csvfile:
        fieldnames = ['timestamp', 'person_id', 'Jitter', 'Shimmer', 'HNR', 'F1', 'F2', 'MFCCs']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

        if not file_exists:
            writer.writeheader()
        
        row_data = {
            'timestamp': datetime.now().isoformat(),
            'person_id': person_id,
            **analysis_results
        }
        writer.writerow(row_data)
    print(f"Voice data for person {person_id} saved to {file_name}.")

def process_all_voice_files(directory='.'):
    """
    Finds all voice files in a directory, analyzes them, and saves the results.
    Voice files are expected to be named like 'v1.wav', 'v2.wav', etc.
    """
    audio_files = glob.glob(os.path.join(directory, 'v*.wav'))
    for audio_path in audio_files:
        match = re.match(r'.*v(\d+)\.wav', audio_path)
        if match:
            person_id = match.group(1)
            print(f"\nProcessing voice for person ID: {person_id}")
            voice_data = analyze_voice_biomarkers(audio_path)
            if voice_data:
                save_voice_to_csv(person_id, voice_data)

if __name__ == '__main__':
    # Process all voice files in the current directory and save the data.
    process_all_voice_files()
